In [1]:
import keras.layers
import unicodedata
import numpy as np
import pandas as pd
from openpyxl.styles.builtins import output
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ctypes

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace


C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)

-2147483648

In [3]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=550000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [4]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [5]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='Mn' )

In [6]:
len(data)

550000

In [7]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w = re.sub(r"([.!?])", r" \1 ", w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [8]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy . <end>'

In [9]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو . <end>'

In [10]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [11]:
max_length=35
batch_size=128

trainer=BpeTrainer(vocab_size=5000)

token_en=Tokenizer(BPE(unk_token="<unk>"))
token_fa=Tokenizer(BPE(unk_token="<unk>"))

token_en.add_special_tokens(["<pad>", "<unk>", "<start>", "<end>"])
token_fa.add_special_tokens(["<pad>", "<unk>", "<start>", "<end>"])
token_en.pre_tokenizer=Whitespace()
token_fa.pre_tokenizer=Whitespace()

token_en.train_from_iterator(df["en"].tolist(), trainer)
token_fa.train_from_iterator(df["fa"].tolist(), trainer)

def encode(tokenizer, text):
    return tokenizer.encode(text).ids


In [12]:
en_seq=[encode(token_en, t) for t in df["en"]]
fa_seq=[encode(token_fa, t) for t in df["fa"]]

en_seq=pad_sequences(en_seq,maxlen=max_length,padding="post",truncating="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post",truncating="post")

inputs=fa_seq[:, :-1]
targets=fa_seq[:, 1:]

decoder_seq_lenght=max_length-1
vocab_size_en=token_en.get_vocab_size()
vocab_size_fa=token_fa.get_vocab_size()

In [13]:
num_layer=4
d_model=256
num_he=8
d_ff=512
dropout_rate=0.1

def positional_encoder(seq_len,d_model):
    positional=np.arange(seq_len)[:,np.newaxis]
    dims=np.arange(d_model)[np.newaxis,:]
    angle_rate=1/np.power(10000,(2*(dims//2))/np.float32(d_model))
    angl=positional*angle_rate
    angl[:,0::2]=np.sin(angl[:,0::2])
    angl[:,1::2]=np.cos(angl[:,1::2])
    return tf.cast(angl[np.newaxis,...],tf.float32)



In [14]:
def feed_forward_n(d_model,d_ff):
    return tf.keras.Sequential([
        Dense(d_ff,activation='relu'),
        Dense(d_model)
    ])

In [15]:
class encoderlayer(tf.keras.layers.Layer):
    def __init__(self,d_model,num_he,d_ff,dropout_rate):
        super().__init__()
        self.mha=MultiHeadAttention(num_heads=num_he,key_dim=d_model//num_he)
        self.ffn=feed_forward_n(d_model,d_ff)
        self.norm1=LayerNormalization(epsilon=0.000001)
        self.norm2=LayerNormalization(epsilon=0.000001)
        self.dropout1=Dropout(dropout_rate)
        self.dropout2=Dropout(dropout_rate)

    def call(self,x,msk,training):
        attn_out=self.mha(query=x,key=x,value=x,attention_mask=msk)
        attn_out=self.dropout1(attn_out,training=training)
        out1=self.norm1(x+attn_out)

        ffn_outp=self.ffn(out1)
        ffn_outp=self.dropout2(ffn_outp,training=training)
        outp2=self.norm2(out1+ffn_outp)

        return outp2

In [16]:
class decoderlayer(tf.keras.layers.Layer):
    def __init__(self,d_model,num_he,d_ff,dropout_rate):
        super().__init__()
        self.mha1=MultiHeadAttention(num_heads=num_he,key_dim=d_model//num_he)
        self.mha2=MultiHeadAttention(num_heads=num_he,key_dim=d_model//num_he)

        self.ffn=feed_forward_n(d_model, d_ff)
        self.norm1=LayerNormalization(epsilon=0.000001)
        self.norm2=LayerNormalization(epsilon=0.000001)
        self.norm3=LayerNormalization(epsilon=0.000001)
        self.dropout1=Dropout(dropout_rate)
        self.dropout2=Dropout(dropout_rate)
        self.dropout3=Dropout(dropout_rate)

    def call(self,x,enc_output,training,ahead_msk,padding_mask):
        attn1=self.mha1(query=x,key=x,value=x,attention_mask=ahead_msk)
        attn1=self.dropout1(attn1,training=training)
        out1=self.norm1(attn1+x)

        attn2=self.mha2(query=out1,key=enc_output,value=enc_output,attention_mask=padding_mask)
        attn2=self.dropout2(attn2,training=training)
        out2=self.norm2(attn2+out1)

        ffn_output=self.ffn(out2)
        ffn_output=self.dropout3(ffn_output,training=training)
        out3=self.norm3(ffn_output+out2)


        return out3


In [17]:
class encoder(tf.keras.layers.Layer):
    def __init__(self,num_layer,d_model,num_he,d_ff,dropout_rate,vocab_size,max_length):
        super().__init__()
        self.d_model=d_model
        self.embedding=Embedding(vocab_size,d_model)
        self.pos_encoding=positional_encoder(max_length,d_model)
        self.enc_layers=[encoderlayer(d_model,num_he,d_ff,dropout_rate) for _ in range(num_layer) ]
        self.dropout=Dropout(dropout_rate)

    def call(self,x,training,msk):
        seq_len=tf.shape(x)[1]
        x=self.embedding(x)
        x*=tf.math.sqrt(tf.cast(self.d_model,tf.float32))
        x+=self.pos_encoding[:,:seq_len,:]
        x=self.dropout(x,training=training)

        for layer in self.enc_layers:
            x=layer(x,training=training,msk=msk)

        return x

In [18]:
class decoder(tf.keras.layers.Layer):
    def __init__(self,num_layer,d_model,num_he,d_ff,dropout_rate,vocab_size,max_length):
        super().__init__()
        self.d_model=d_model
        self.embedding=Embedding(vocab_size,d_model)
        self.pos_encoding=positional_encoder(max_length,d_model)
        self.de_layer=[decoderlayer(d_model,num_he,d_ff,dropout_rate) for _ in range(num_layer)]
        self.dropout=Dropout(dropout_rate)


    def call(self,x,enc_output,training,ahead_msk,padding_mask):
        seq_len=tf.shape(x)[1]
        x=self.embedding(x)
        x*=tf.math.sqrt(tf.cast(self.d_model,tf.float32))
        x+=self.pos_encoding[:,:seq_len,:]
        x=self.dropout(x,training=training)
        for layer in self.de_layer:
            x=layer(x,enc_output,training=training,ahead_msk=ahead_msk,padding_mask=padding_mask)

        return x

In [19]:
def padding_msk(seq):
    mask=tf.cast(tf.not_equal(seq,0),tf.float32)
    return mask[:,tf.newaxis,tf.newaxis,:]

In [20]:
def ahead_mask(size):
    mask=tf.linalg.band_part(tf.ones((size,size)),-1,0)
    return mask

In [21]:
class translator_model(tf.keras.Model):
    def __init__(self,num_layer,d_model,num_he,d_ff,dropout_rate,max_length,vocab_size_fa,vocab_size_en):
        super().__init__()
        self.encoder=encoder(num_layer,d_model,num_he,d_ff,dropout_rate,vocab_size_en,max_length)
        self.decoder=decoder(num_layer,d_model,num_he,d_ff,dropout_rate,vocab_size_fa,max_length)
        self.final_layer=Dense(vocab_size_fa,activation='softmax')


    def call(self,inputs,training=False):
        enc_input,de_input=inputs
        enc_padding_msk=padding_msk(enc_input)
        de_padding_msk=padding_msk(enc_input)
        l_ahead_mask=ahead_mask(tf.shape(de_input)[1])
        de_target_padding_mask=padding_msk(de_input)
        combined_mask=tf.minimum(de_target_padding_mask,l_ahead_mask)

        enc_output=self.encoder(enc_input,training=training,msk=enc_padding_msk)
        de_output=self.decoder(de_input,enc_output,training=training,ahead_msk=combined_mask,padding_mask=de_padding_msk)

        F_output=self.final_layer(de_output)
        return F_output


In [22]:
model=translator_model(num_layer,d_model,num_he,d_ff,dropout_rate,max_length,vocab_size_fa,vocab_size_en)

In [23]:
model.summary()# is normal

Model: "translator_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (encoder)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (decoder)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
print(model.encoder.pos_encoding.shape)

(1, 35, 256)


In [25]:
import inspect
print(inspect.signature(translator_model.__init__))

(self, num_layer, d_model, num_he, d_ff, dropout_rate, max_length, vocab_size_fa, vocab_size_en)


In [26]:
n=len(en_seq)
train_end=int(n*0.85)
val_end=int(n * 0.95)

en_train=en_seq[:train_end]
en_val=en_seq[train_end:val_end]
en_test=en_seq[val_end:]

fa_train=fa_seq[:train_end]
fa_val=fa_seq[train_end:val_end]
fa_test=fa_seq[val_end:]

inputs_train=fa_train[:, :-1]
targets_train=fa_train[:, 1:]

inputs_val=fa_val[:, :-1]
targets_val=fa_val[:, 1:]

inputs_test=fa_test[:, :-1]
targets_test=fa_test[:, 1:]

train_ds=tf.data.Dataset.from_tensor_slices(((en_train, inputs_train), targets_train)).shuffle(30000).batch(
    batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_val, inputs_val), targets_val)).batch(batch_size).prefetch(
    tf.data.AUTOTUNE)
test_ds=tf.data.Dataset.from_tensor_slices(((en_test, inputs_test), targets_test)).batch(batch_size).prefetch(
    tf.data.AUTOTUNE)

In [27]:
loss_ob=tf.keras.losses.SparseCategoricalCrossentropy(reduction="none",from_logits=False)

In [28]:
def msk_loss(y_true,y_pred):

    loss=loss_ob(y_true,y_pred)

    mask=tf.cast(tf.not_equal(y_true,0),dtype=loss.dtype)
    loss=loss*mask
    return tf.reduce_sum(loss)/tf.reduce_sum(mask)

In [29]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),metrics=[],loss=msk_loss)

In [30]:
early=EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


In [31]:
print(f"number of english sentences: {len(en_s)}")
print(f"number of persian sentences: {len(fa_s)}")
print(f"are they equal? {len(en_s)==len(fa_s)}")

number of english sentences: 612086
number of persian sentences: 612086
are they equal? True


In [32]:
import random
sample_idx=random.sample(range(len(en_s)), 15)
for i in sample_idx:
    print(f"EN: {en_s[i]}")
    print(f"FA: {fa_s[i]}")
    print("---")

EN: when did you contact with jang?
FA: كي با جانگ تماس گرفتي؟
---
EN: then maybe you don't know better,
FA: پس شايد بهتر نمي دوني
---
EN: you followed him here and doing dangerous thing now
FA: تو با وجود تمام خطرات دنبال اون اومدي به اينجا
---
EN: who the fuck said that shit .
FA: كدوم عوضي اي اين كس شعرا را گفته .
---
EN: my son , finally having the noodle dream .
FA: پسر من ، بالاخره خواب ماکاروني ديد .
---
EN: lt�s because l�m a jew .
FA: چون من يهودي هستم .
---
EN: but why does he want to mention it again?
FA: اما چرا دوباره اين موضوع رو مطرح كرد؟
---
EN: is there any other way to plant the seeds other than to cover with dirt?
FA: آیا راه دیگه ای علاوه بر پوشاندن بذر بوسیله ی خاک می شناسید؟
---
EN: all 9's, right?
FA: همش 9 ميليمتريه ، درسته ؟
---
EN: is what he said true?
FA: اون چيز هايي که گفت راست بود؟
---
EN: or a miracle .
FA: يا يک معجزه .
---
EN: smoked the last two back in salt lake .
FA: 2تاي اخريو در درياچه نمک کشيدم لعنتي .
---
EN: walter, what are you doing?
FA: والت

In [33]:
en_lengths = [len(encode(token_en, t)) for t in df["en"].tolist()]
fa_lengths = [len(encode(token_fa, t)) for t in df["fa"].tolist()]

en_over = sum(1 for l in en_lengths if l > max_length)
fa_over = sum(1 for l in fa_lengths if l > max_length)

print(f"current max_length: {max_length}")
print(f"english sentences longer than max_length: {en_over} ({en_over / len(en_lengths) * 100:.1f}%)")
print(f"persian sentences longer than max_length: {fa_over} ({fa_over / len(fa_lengths) * 100:.1f}%)")
print(
    f"english mean length: {np.mean(en_lengths):.1f} | 95th percentile: {np.percentile(en_lengths, 95):.1f} | max: {np.max(en_lengths)}")
print(
    f"persian mean length: {np.mean(fa_lengths):.1f} | 95th percentile: {np.percentile(fa_lengths, 95):.1f} | max: {np.max(fa_lengths)}")


long_idx_en = [i for i, l in enumerate(en_lengths) if l > max_length][:3]

for i in long_idx_en:
    original_ids = encode(token_en, df["en"].iloc[i])
    padded = pad_sequences([original_ids], maxlen=max_length, padding="post")[0]

    print(f"original sentence: {df['en'].iloc[i]}")
    print(f"actual token count: {len(original_ids)}")
    print(f"original tokens first 10: {[token_en.id_to_token(t) for t in original_ids[:10]]}")
    print(f"after pad_sequences with max_length={max_length}: {[token_en.id_to_token(t) for t in padded if t != 0]}")
    print(f"is <start> still present? {token_en.token_to_id('<start>') in padded}")
    print("---")

current max_length: 35
english sentences longer than max_length: 19 (0.0%)
persian sentences longer than max_length: 54 (0.0%)
english mean length: 11.0 | 95th percentile: 19.0 | max: 43
persian mean length: 10.7 | 95th percentile: 19.0 | max: 47
original sentence: <start> three measures of gordons , one of vodka , half a measure of china lillet  .  shake up , add some ice and a thin slice of lemon  . <end>
actual token count: 40
original tokens first 10: ['<start>', 'three', 'measure', 's', 'of', 'gor', 'don', 's', ',', 'one']
after pad_sequences with max_length=35: ['gor', 'don', 's', ',', 'one', 'of', 'v', 'od', 'ka', ',', 'half', 'a', 'measure', 'of', 'china', 'li', 'll', 'et', '.', 'shake', 'up', ',', 'add', 'some', 'ice', 'and', 'a', 'thin', 'sli', 'ce', 'of', 'le', 'mon', '.', '<end>']
is <start> still present? False
---
original sentence: <start> madam jami has rounded up the chiefs from hanju and wungju han joo: current seoul and kyoung gi regions woong joo: current gong joo <

In [34]:
long_idx_fa = [i for i, l in enumerate(fa_lengths) if l > max_length][:3]

for i in long_idx_fa:
    original_ids = encode(token_fa, df["fa"].iloc[i])
    padded = pad_sequences([original_ids], maxlen=max_length, padding="post")[0]

    print(f"original sentence: {df['fa'].iloc[i]}")
    print(f"actual token count: {len(original_ids)}")
    print(f"original tokens first 10: {[token_fa.id_to_token(t) for t in original_ids[:10]]}")
    print(f"after pad_sequences with max_length={max_length}: {[token_fa.id_to_token(t) for t in padded if t != 0]}")
    print(f"is <start> still present? {token_fa.token_to_id('<start>') in padded}")
    print("---")

original sentence: <start> در حقیقت کاری که اونا می کنن اینه که می خورن و می خورن و بعد تخلیه می کنن ، بعد دوباره می خورن این کسی را یادت نمی اندازه  . <end>
actual token count: 39
original tokens first 10: ['<start>', 'در', 'حقیقت', 'کاری', 'که', 'اونا', 'می', 'کنن', 'اینه', 'که']
after pad_sequences with max_length=35: ['که', 'اونا', 'می', 'کنن', 'اینه', 'که', 'می', 'خور', 'ن', 'و', 'می', 'خور', 'ن', 'و', 'بعد', 'ت', 'خل', 'یه', 'می', 'کنن', '،', 'بعد', 'دوباره', 'می', 'خور', 'ن', 'این', 'کسی', 'را', 'ی', 'ادت', 'نمی', 'اندازه', '.', '<end>']
is <start> still present? False
---
original sentence: <start> ما بايد به شهر هن-تو، اوك-جو،دونگ-يه، جول-بون، يانگ-مك،كو-دا،وكه-ما پيك بفرستيم <end>
actual token count: 39
original tokens first 10: ['<start>', 'ما', 'بايد', 'به', 'شهر', 'هن', '-', 'تو', '،', 'او']
after pad_sequences with max_length=35: ['شهر', 'هن', '-', 'تو', '،', 'او', 'ك', '-', 'جو', '،', 'دونگ', '-', 'يه', '،', 'جول', '-', 'بون', '،', 'يانگ', '-', 'مك', '،', 'كو', '-', 'دا'

In [35]:
l=ReduceLROnPlateau(monitor="val_loss",patience=2,min_lr=0.000001,verbose=1,factor=0.5)

In [36]:
history=model.fit(train_ds,epochs=120,validation_data=val_ds,verbose=2,callbacks=[early,l])

Epoch 1/120
3653/3653 - 3724s - 1s/step - loss: 4.1200 - val_loss: 3.3483 - learning_rate: 0.0010
Epoch 2/120
3653/3653 - 3208s - 878ms/step - loss: 3.1736 - val_loss: 2.9991 - learning_rate: 0.0010
Epoch 3/120
3653/3653 - 9348s - 3s/step - loss: 2.9113 - val_loss: 2.8528 - learning_rate: 0.0010
Epoch 4/120
3653/3653 - 4038s - 1s/step - loss: 2.7688 - val_loss: 2.7727 - learning_rate: 0.0010
Epoch 5/120
3653/3653 - 3756s - 1s/step - loss: 2.6690 - val_loss: 2.7224 - learning_rate: 0.0010
Epoch 6/120
3653/3653 - 2949s - 807ms/step - loss: 2.5960 - val_loss: 2.6849 - learning_rate: 0.0010
Epoch 7/120
3653/3653 - 3015s - 825ms/step - loss: 2.5370 - val_loss: 2.6541 - learning_rate: 0.0010
Epoch 8/120
3653/3653 - 2997s - 820ms/step - loss: 2.4906 - val_loss: 2.6274 - learning_rate: 0.0010
Epoch 9/120
3653/3653 - 3011s - 824ms/step - loss: 2.4457 - val_loss: 2.6110 - learning_rate: 0.0010
Epoch 10/120
3653/3653 - 3011s - 824ms/step - loss: 2.4086 - val_loss: 2.6083 - learning_rate: 0.0010
E

In [41]:
model.save_weights('translator.weights.h5')

In [26]:
def translate(s):
    s=preprossing(s)
    en_seq_input=[token_en.encode(s).ids]
    en_seq_input=pad_sequences(en_seq_input,maxlen=max_length, padding="post",truncating="post")
    encoder_input=tf.convert_to_tensor(en_seq_input)

    s_token_id=token_fa.token_to_id("<start>")
    end_token_id=token_fa.token_to_id("<end>")

    decoder_input=[s_token_id]
    for i in range(max_length):
        de_inputs_tensor=tf.expand_dims(decoder_input,0)

        prediction=model((encoder_input,de_inputs_tensor),training=False)
        prediction=prediction[:,-1,:]
        prediction_id=tf.argmax(prediction,axis=-1).numpy()[0]
        if prediction_id==end_token_id:
            break
        decoder_input.append(int(prediction_id))


        output_tokens=[reverse_fa.get(idx,"") for idx in decoder_input[1:]]
        return " ".join(output_tokens).strip()


In [ ]:
print(tf.config.list_physical_devices('GPU'))